In [0]:
%pip install -U -qqqq backoff databricks-sdk databricks_vectorsearch
%pip install --upgrade openai replicate
dbutils.library.restartPython()

In [0]:
from config import DeployConfig

In [0]:
dbutils.widgets.text("config_path", "./config/env_variables.yml")
config_path = dbutils.widgets.get("config_path")
cfg = DeployConfig.from_yaml(config_path)

In [0]:
vs_index_path = getattr(cfg, f"vs_index").path
vs_index_endpoint = getattr(cfg, f"vs_index").endpoint
image_table_path = getattr(cfg, f"image_table").path
image_gen_model_path = getattr(cfg, f"image_gen_model_path").path
agent_endpoint_name = getattr(cfg, f"agent_endpoint_name")
image_gen_output_path = getattr(cfg, f"image_gen_output").path

#AGENT BUILD

In [0]:
import base64
import json
import mlflow
import pandas as pd
import os
import replicate
import requests
import tempfile
import time
import uuid

from databricks.sdk import WorkspaceClient
from databricks.vector_search.index import VectorSearchIndex
from databricks.vector_search.client import VectorSearchClient
from io import BytesIO
from mlflow.deployments import get_deploy_client
from mlflow.entities import SpanType
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

In [0]:
os.environ['CLIENT_ID'] = dbutils.secrets.get("shovakeemian-scope", "shovakeemian-sp-client-id")
os.environ['CLIENT_SECRET'] = dbutils.secrets.get("shovakeemian-scope", "shovakeemian-sp-client-secret")
os.environ['REPLICATE_API_TOKEN'] = dbutils.secrets.get("jssandom-scope", "replicate-key")
os.environ['UC_VOLUME_PATH'] = image_gen_output_path

In [0]:
class PetAgent(mlflow.pyfunc.PythonModel):

    def __init__(self):
        self.CLIENT_ID = os.environ.get("CLIENT_ID") 
        self.CLIENT_SECRET = os.environ.get("CLIENT_SECRET")

        raw_path = os.environ.get("UC_VOLUME_PATH")
        # Strip dbfs
        if raw_path.startswith("dbfs:"):
            raw_path = raw_path[len("dbfs:"):]
        self.image_gen_output_path = raw_path
       

    def load_context(self, context):
        from transformers import CLIPProcessor, CLIPModel

        self.vsc=VectorSearchClient(    
            workspace_url="https://e2-demo-field-eng.cloud.databricks.com/",
            service_principal_client_id=self.CLIENT_ID,
            service_principal_client_secret=self.CLIENT_SECRET
        )

        # WorkspaceClient for UC Volume uploads
        self.w = WorkspaceClient(
            host="https://e2-demo-field-eng.cloud.databricks.com/",
            client_id=self.CLIENT_ID,
            client_secret=self.CLIENT_SECRET
        )

        self.model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
        self.processor= CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
        self.brand_image_path = context.artifacts.get("brand_image")

        with open(self.brand_image_path, "rb") as f:
            self.brand_image = f.read()


    @mlflow.trace(name="compute_text_embedding", span_type=SpanType.EMBEDDING, attributes={"model": "clip-vit-large-patch14"})
    def _get_text_embedding(self, text):
      """
      computes the text embedding for a given text.
      """
      # Want to change this to call the serving endpoint when ai_query can take params of a pyfunc.
      inputs = self.processor(text=text, return_tensors="pt", padding=True)
      text_features = self.model.get_text_features(**inputs)
      return text_features.detach().numpy().tolist()[0]


    @mlflow.trace(name="pet_image_vs_imageemb_lookup", span_type=SpanType.RETRIEVER, attributes={"model": "clip-vit-large-patch14", "vs_index": vs_index_path})
    def _vector_search_retrieval(self, query, top_k: int = 3):
      text_embed_query=self._get_text_embedding(query)
      index = self.vsc.get_index(endpoint_name=vs_index_endpoint, index_name=vs_index_path)
      vs_output = index.similarity_search(columns=["id", "model_input", "path"], query_vector=text_embed_query, num_results=int(top_k))
      return vs_output


    @mlflow.trace(name="pet_id_image_lookup", span_type=SpanType.RETRIEVER, attributes={"table": image_table_path})
    def _get_image_candidates(self, vs_output, limit: int = 3) -> list[str]:
        """
        Extract top-k UC Volume paths from VS output (no decoding).
        """
        rows = (vs_output or {}).get("result", {}).get("data_array", []) or []
        paths = []
        for r in rows[:limit]:
            # expected order: ["id","model_input","path"]
            p = (r[2] or "").replace("dbfs:/", "/")
            if p:
                paths.append(p)
        return paths
    

    def _read_uc_bytes(self, path: str) -> bytes:
        path = path.replace("dbfs:/", "/")
        return self.w.files.download(path).contents.read()


    @mlflow.trace(name="replicate_image_generation", span_type=SpanType.TOOL)
    def _replicate_image_generation(self, pet_image: str, brand_image: bytes) -> str:
        # Decode inputs
        pet_image_bytes = base64.b64decode(pet_image, validate=True)
        brand_image_bytes = bytes(brand_image)

        # Use unique temp files to avoid collisions under concurrent requests
        with tempfile.NamedTemporaryFile(suffix=".png") as pet_tmp, \
            tempfile.NamedTemporaryFile(suffix=".png") as brand_tmp:

            pet_tmp.write(pet_image_bytes);   pet_tmp.flush()
            brand_tmp.write(brand_image_bytes); brand_tmp.flush()

            # Open for reading only within the call
            with open(pet_tmp.name, "rb") as f_pet, open(brand_tmp.name, "rb") as f_brand:
                try:
                    out = replicate.run(
                        "flux-kontext-apps/multi-image-kontext-max",
                        input={
                            "seed": 42,
                            "prompt": (
                                "Create a product advertisement for pet food with the images provided. "
                                "Show the pet from pet_image interacting naturally with the bag of pet food "
                                "from petfood_brand. Keep the pet's pose natural and the original environment. "
                                "No new text; only text on the bag. Do not take food out of the bag."
                            ),
                            "aspect_ratio": "1:1",
                            "input_image_1": f_pet,
                            "input_image_2": f_brand,
                            "output_format": "png",
                            "safety_tolerance": 2,
                        },
                    )
                    output_bytes = out.read()  # raises if stream fails
                except Exception as e:
                    # Let the caller/tool handle the error string; no retries here
                    raise RuntimeError(f"Replicate generation failed: {e}") from e

        return base64.b64encode(output_bytes).decode("utf-8")
    

    @mlflow.trace(name="upload_final_image_uc_volume", span_type=SpanType.TOOL, attributes={"target": "/Volumes"})
    def _upload_to_uc_volume(self, img_b64: str, query: str) -> str:
        """
        Decodes base64 image and uploads to UC Volume using WorkspaceClient.
        Returns the absolute UC volume path (e.g., /Volumes/<catalog>/<schema>/<volume>/<file>.png).
        """
        # Make a stable, unique filename
        safe_query = "".join(c for c in (query or "ad") if c.isalnum() or c in ("-", "_"))[:40]
        ts = int(time.time())
        fname = f"{safe_query or 'ad'}_{ts}_{uuid.uuid4().hex[:8]}.png"

        dest_path = f"{self.image_gen_output_path}{fname}" 
        img_bytes = base64.b64decode(img_b64, validate=True)

        # Upload via Files API (WorkspaceClient)
        # Accepts file-like object; we pass an in-memory BytesIO
        with BytesIO(img_bytes) as fh:
            self.w.files.upload(dest_path, fh, overwrite=False)

        return dest_path
    

    def _get_replicate_toggle(self, params):
      # Default is False
      if isinstance(params, dict):
          return bool(params.get("replicate_toggle", False))
      if isinstance(params, bool):
          return params
      return False


    @mlflow.trace(name="ad-image-agent")
    def predict(self, context, model_input, params=None):
        params = params or {}
        replicate_toggle = self._get_replicate_toggle(params)
        num_candidates   = int(params.get("num_candidates", 3))
        seed_index       = int(params.get("seed_index", 0))

        if isinstance(model_input, pd.DataFrame):
            query = model_input["model_input"].iloc[0]
        elif isinstance(model_input, dict):
            query = model_input.get("model_input", "")
        elif isinstance(model_input, str):
            query = model_input
        else:
            raise ValueError("Unsupported input type")

        # --- retrieval (top-k) ---
        vs_output  = self._vector_search_retrieval(query=query, top_k=num_candidates)
        seed_paths = self._get_image_candidates(vs_output, limit=num_candidates)

        # seed index
        chosen_seed_index = 0 if not seed_paths else max(0, min(seed_index, len(seed_paths) - 1))
        final_ad_path = None

        # --- optional single final generation --- #
        if replicate_toggle and seed_paths:
            # read chosen seed, base64 encode for replicate tool
            pet_bytes = self._read_uc_bytes(seed_paths[chosen_seed_index])
            pet_b64   = base64.b64encode(pet_bytes).decode("utf-8")

            final_ad_b64 = self._replicate_image_generation(
                pet_image=pet_b64,
                brand_image=self.brand_image
            )
            # Upload to UC Volume (your existing helper) and return the UC path
            final_ad_path = self._upload_to_uc_volume(final_ad_b64, query=query)

        resp_obj = {
            "query": query,
            "seed_paths": seed_paths,
            "chosen_seed_index": chosen_seed_index,
            "final_uc_path": final_ad_path
        }
        return json.dumps(resp_obj)

In [0]:
agent = PetAgent()

class DummyContext:
    artifacts = {
        "brand_image": f"/Volumes/ml_shovakeemian/feip/petfood_brand_creatives/BricksV1.png"
    }
agent.load_context(DummyContext)

In [0]:
response = agent.predict(
  model_input='black forest cat', context=None, params={'replicate_toggle': False, 'num_candidates': 3})

In [0]:
# return response as dict
json.loads(response)

## DEPLOY

In [0]:
import pandas as pd

example_input = pd.DataFrame({
    "model_input": ["black forest cat"]
})

In [0]:
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec, ParamSchema, ParamSpec

input_schema = Schema([
    ColSpec("string", "model_input")
])

output_schema = Schema([
    ColSpec("string", "response_json")
])


params_schema = ParamSchema([
    ParamSpec("replicate_toggle", "boolean", False),
    ParamSpec("num_candidates", "integer", 3),
    ParamSpec("seed_index", "integer", 0),
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema, params=params_schema)

In [0]:
pip_requirements=[
  "--extra-index-url https://download.pytorch.org/whl/cu121", 
  "openai==1.101.0",
  "replicate==1.0.7",
  "databricks-vectorsearch==0.57",
  "mlflow==2.19.0",
  "setuptools<70.0.0", 
  "torch==2.3.1+cu121", 
  "accelerate==0.31.0", 
  "astunparse==1.6.3", 
  "bcrypt==3.2.0", 
  "boto3==1.34.39", 
  "configparser==5.2.0", 
  "defusedxml==0.7.1", 
  "dill==0.3.6",
   "google-cloud-storage==2.10.0", 
   "ipython==8.15.0", 
   "lz4==4.3.2", 
   "nvidia-ml-py==12.555.43", 
   "optree==0.12.1", 
   "pandas==1.5.3",
   "numpy==1.23.5",
   "pyopenssl==23.2.0", 
   "pytesseract==0.3.10", 
   "scikit-learn==1.3.0", 
   "sentencepiece==0.1.99", 
   "torchvision==0.18.1+cu121", 
   "transformers==4.41.2",
   "https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.3cxx11abiFALSE-cp311-cp311-linux_x86_64.whl"
   ]

In [0]:
from mlflow.models.resources import DatabricksVectorSearchIndex

resources = [DatabricksVectorSearchIndex(index_name=vs_index_path)]

with mlflow.start_run():
  mlflow.pyfunc.log_model(
    "agent", 
    python_model=PetAgent(),
    resources=resources,
    signature=signature,
    pip_requirements=pip_requirements,
    input_example=example_input,
    artifacts={
      "brand_image": "/Volumes/ml_shovakeemian/feip/petfood_brand_creatives/BricksV1.png"},
  )

In [0]:
run_id = mlflow.last_active_run().info.run_id

In [0]:
mlflow.set_registry_uri("databricks-uc")

# Register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=f"runs:/{run_id}/agent", name=image_gen_model_path
)

In [0]:
test_loaded=mlflow.pyfunc.load_model(f"models:/{image_gen_model_path}/13")

In [0]:
df = pd.DataFrame({"model_input": ["French Bulldog"]})

response = test_loaded.predict(df, params={'replicate_toggle': True})

In [0]:
json.loads(response)

In [0]:
from mlflow.deployments import get_deploy_client

client = get_deploy_client("databricks")
# response = client.create_endpoint(
#     name=agent_endpoint_name,
#     config={
#         "served_entities": [
#             {
#                 "name": agent_endpoint_name,
#                 "entity_name": image_gen_model_path,
#                 "entity_version": uc_registered_model_info.version,
#                 "workload_size": "Small",
#                 "scale_to_zero_enabled": True,
#                 "workload_type": "GPU_SMALL",
#                 "environment_vars": {
#                     "CLIENT_ID": "{{secrets/shovakeemian-scope/shovakeemian-sp-client-id}}",
#                     "CLIENT_SECRET": "{{secrets/shovakeemian-scope/shovakeemian-sp-client-secret}}",
#                     "REPLICATE_API_TOKEN": "{{secrets/jssandom-scope/replicate-key}}",
#                 }
#             }
#         ],
#     }
# )

# Update the existing endpoint
response = client.update_endpoint(
    endpoint=agent_endpoint_name, # existing endpoint name
    config={
        "served_entities": [
            {
                "name": agent_endpoint_name,
                "entity_name": image_gen_model_path,
                "entity_version": uc_registered_model_info.version,
                "workload_type": "GPU_SMALL",
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
                "environment_vars": {
                    "CLIENT_ID": "{{secrets/shovakeemian-scope/shovakeemian-sp-client-id}}",
                    "CLIENT_SECRET": "{{secrets/shovakeemian-scope/shovakeemian-sp-client-secret}}",
                    "REPLICATE_API_TOKEN": "{{secrets/jssandom-scope/replicate-key}}",
                    "UC_VOLUME_PATH": image_gen_output_path,
                }
            }
        ]
    }
)

print(response)